# 56. Random-label fitting, on the revised streams

## Why this exists

The paper says three no-training diagnostics report a loss of plasticity while
direct measurements say the model learns at least as well as before. Every
diagnostic and most of the direct measurements now sit on the revised design of
nine groups of twenty classes. Random-label fitting does not. It was measured
on `indicator_stream_v1`, a retired training run of eight tasks of ten classes
whose model files the current class split cannot load.

So the random-label row in the paper still carries a "different training run"
caveat. This notebook removes it.

## What it measures

The direct probe asks whether the model can still learn something useful.
Random-label fitting asks something narrower: can it still change at all.

The model is given 2,000 images whose labels were shuffled at random. There is
no pattern to find, so nothing can be transferred from what the stream taught
it. All that is left is raw capacity to move. How far the training loss falls
is the measurement.

Lower loss means it memorised more, so **lower means more capacity to change**.
That is the opposite reading from the direct probe, where higher is better, and
it is the easiest thing to get backwards when reading the table.

## What stays at notebook 41's values

The label seed, 2,000 images, 600 steps, the same three learning rates, and the
same rule for choosing among them: keep the run with the lowest final training
loss. The rate is picked on the same number the result is reported from, which
is why the number is a best-achievable fit rather than a score.

## What changes, and why

The study. Everything else follows from it.

The probe task is group 9, the twenty classes the stream never touches. In the
retired design the random-label task was simply one that happened to be unused.
Here it is held out on purpose.

Twenty classes instead of ten. Chance for a random-label fit is a loss of
ln(20) = 2.996 rather than ln(10) = 2.303, and twenty wrong labels are harder
to memorise than ten on the same budget. The budget is left alone anyway, so
that this run and notebook 41's choose from the same grid, and section 6 stops
the session if the first stream ends near chance. A probe that never memorised
anything is measuring the step budget and not the model, and no decision rule
may be applied to it.

All three orderings at five seeds, instead of one ordering at three seeds.
Ordering A ran on 22 November and fell past its floor on all five streams.
Orderings B and C are the second pass, asking whether the order the nine
groups arrive in changes that.

Boundary 0 is measured again for each ordering. The starting generator is
rebuilt from the seed, so the three orderings of one seed should start from the
same model, which makes those readings a repeat measurement of the same
configuration as well as the comparison point each stream's floor is built
from. Notebook 47 found such repeats differing by about a ninth of the effect
it was measuring, so a difference here is worth looking at rather than
assuming.

## Cost

Fifteen streams, two points, three rates, five repetitions is 450 runs of 600
steps. October's random-label runs took a median of 1.34 minutes each, so the
whole grid is about ten hours plus model downloads.

Ordering A's 150 runs are already in the resume file and are matched one at a
time, so this session runs the remaining 300, about seven hours. Stopping part
way and continuing another day repeats nothing.

## Where to run it

Either machine. Seven hours is longer than Colab reliably holds, so a rented
GPU is the safer one for the whole grid; on Colab, run a few streams at a time
by editing `ORDERS` and `SEEDS` and let the resume file join the sessions up.

The notebook finds its own checkout. If the working directory is inside one it
uses that; otherwise it clones the repository, so uploading this file on its
own and running it from the top works with no manual clone.

It puts the code and the data in different places on purpose. The code goes on
the machine's own disk, under `CODE_ROOT`, which defaults to the home
directory. The large downloads go on `DATA_DISK`, which defaults to
`/workspace`, a rented pod's mounted volume, and falls back to the checkout
where there is no volume.

The reason is that git cannot clone onto a network volume. On RunPod a clone
into `/workspace` stops with `chmod on .git/config.lock failed: Operation not
permitted`, because the volume maps the container's root user to one that owns
nothing. Reading and writing ordinary data files there works; only git does
not. Both folders can be moved by setting `CODE_ROOT` or `DATA_DISK` in the
environment.

Two things to set up before starting it.

The three R2 credentials, as environment variables. Section 3 stops the
notebook if one is missing.

Disk. Each end-of-stream model is about 550 MB and this session reads ten of
them, which is 5.5 GB, plus about 240 MB of Tiny ImageNet. That goes on
`DATA_DISK`, so give the volume 20 GB. The checkout itself is about 30 MB and
fits on any container disk.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

# This file names a checkout of the repository: it is the ten-group class
# split, which the revised study needs and the older runs do not have.
MARKER = "uncle/task_partition_10x20.json"

# Where a checkout goes when this notebook is opened on its own, which is the
# normal case on both Colab and a rented pod.
#
# Not the mounted volume. Cloning needs a filesystem where git can set
# permissions, and a network volume usually is not one: on RunPod, cloning
# into /workspace fails with "chmod on .git/config.lock failed: Operation not
# permitted", because the volume maps the container's root user to a user that
# owns nothing. Ordinary reading and writing of data files there is fine; it is
# only git that cannot work. So the code goes on the machine's own disk, which
# is always writable, and the large downloads go on the volume. The checkout is
# about 30 MB, so the machine's disk is big enough whatever the pod's size.
CODE_ROOT = (Path("/content") if IN_COLAB
             else Path(os.environ.get("CODE_ROOT", Path.home())))

# A checkout at or above the working directory wins, so a machine where the
# repository was cloned by hand and Jupyter started inside it is left alone.
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / MARKER).exists()), None)
if REPO is None:
    # Nothing to work from, so fetch one. The notebook runs at the current
    # main rather than a pinned commit: the revised design was added after the
    # earlier runs, so an older checkout would not contain it.
    REPO = CODE_ROOT / "path-probe"
    try:
        if not REPO.exists():
            subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    except subprocess.CalledProcessError as failed:
        # The usual cause is a filesystem git cannot set permissions on, which
        # the message above this traceback names. Point somewhere it can.
        raise RuntimeError(
            f"git failed in {REPO}. Read the git error printed above this. If "
            f"it mentions chmod or permissions, the folder is on a mount git "
            f"cannot write to: set CODE_ROOT in the environment to a folder on "
            f"the machine's own disk, such as /root, and run this cell again.") from failed
    if not (REPO / MARKER).exists():
        raise FileNotFoundError(f"{REPO} has no {MARKER}.")

# Where Tiny ImageNet and the downloaded models go, about 6 GB together. A
# rented pod's volume is the roomy disk and it survives the pod stopping, so a
# session started again does not download them twice. Writing data files there
# works even though git cannot. Where there is no volume this falls back to the
# checkout, which on a development machine may already hold the images.
DATA_DISK = Path(os.environ.get("DATA_DISK", "/workspace"))
if not DATA_DISK.is_dir():
    DATA_DISK = REPO

# Installed on whatever machine this is. A rented GPU image usually carries
# torch and nothing else, and the imports below need both of these.
for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

import torch
from tqdm.auto import tqdm
from uncle import checkpoint
from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import HyperNetwork, build_target
from uncle.probe import (RandomLabels, best_of_grid, noise_floor, probe_once,
                         snapshot, unchanged)
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.trainer import UnCLe

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| device:", GPU_NAME)
if DEVICE == "cpu":
    print("No GPU. This notebook trains 150 forks of 600 steps; it needs one.")

# The same settings the other probe notebooks use. Left off, these let the
# library pick a different matrix algorithm from one call to the next, which
# would add spread the noise floor cannot see.
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 2. Settings

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# All three orderings at five seeds. Ordering A ran on 22 November and its 150
# runs are in the resume file under this same RUN_LABEL, so they are matched
# one at a time and skipped: this session runs the 300 that are left, about
# seven hours.
#
# Do not give this run a new RUN_LABEL. Sharing the file with ordering A is the
# point, and the protocol below is unchanged, so the runs are comparable.
ORDERS = ["order_A", "order_B", "order_C"]
SEEDS = [0, 1, 2, 3, 4]

# 0 is the untrained generator, which the seed rebuilds exactly, so it needs no
# file. 9 is the end of the nine-group stream. Boundary 10 is the model after
# the unlearn, which is a different question and does not belong here.
POINTS = [0, 9]

# Names this protocol. Any change below that would make the numbers
# incomparable needs a new label, because finished runs are matched by identity
# and a shared file would quietly mix two protocols.
RUN_LABEL = "revised_20class_random2000"

RESUME_KEY = f"uncle/random_labels/{STUDY_ID}/{RUN_LABEL}.jsonl"
SAVE_EVERY = 10

# The held-out group. The stream never trains on it, so the generator has no
# code for it and the probe can create a fresh one.
RANDOM_LABEL_TASK = PROBE_TASK
CLASSES_HINT = 20

# Held at notebook 41's values. The label seed fixes which wrong label each
# image gets, and every probe point shares that one draw: a fresh draw per
# point would make the points incomparable.
RANDOM_LABEL_SEED = 1020
RANDOM_LABEL_IMAGES = 2000     # 100 per class across the twenty
RANDOM_LABEL_STEPS = 600       # about 19 passes over 2,000 images
SPLIT_SEED = 606

# A model late in a stream may need a different rate than a fresh one, so the
# probe tries all three and keeps the best. A single fixed rate would confuse a
# rate mismatch with lost plasticity. In the retired design 1e-3 was never the
# one chosen at either point; it is kept so the two runs choose from the same
# set.
RATES = [1e-4, 3e-4, 1e-3]
BATCH_SIZE = 64

# Equal counts at both points, so the floor falls as one over the square root
# of the count. Notebook 41 raised these from 5 and 2 to 5 and 5 because the
# 2 was what made one seed's floor as wide as 0.42.
REPETITIONS_AT_ZERO = 5
REPETITIONS_LATER = 5

# (name, generator frozen, code trains). The code-only and generator-only arms
# answered a different question, whether the 32-number code earns its place,
# and that was settled in October.
ARMS = [("full", False, True)]

# How far below chance the fit has to land before the decision rule may be
# applied. A loss sitting at chance means the budget was too small, not that
# capacity is unchanged.
CHANCE_MARGIN = 0.20

# Both on DATA_DISK, which section 1 set: the mounted volume on a rented pod,
# and the checkout itself where there is no volume, as on Colab and on a
# development machine that already holds the images.
DATA_ROOT = DATA_DISK / "data/tiny-imagenet-200"
CACHE = DATA_DISK / "random-labels"
CACHE.mkdir(parents=True, exist_ok=True)
print("Code:", REPO, "| images:", DATA_ROOT, "| downloads:", CACHE)

RUNS = [(order, seed) for order in ORDERS for seed in SEEDS]
TOTAL = len(RUNS) * len(POINTS) * len(RATES) * REPETITIONS_AT_ZERO * len(ARMS)
print(f"{len(RUNS)} streams x {len(POINTS)} points x {len(RATES)} rates x "
      f"{REPETITIONS_AT_ZERO} repetitions = {TOTAL} runs")
print(f"At the 1.34 minutes October's random-label runs took, about "
      f"{TOTAL * 1.34 / 60:.1f} hours, plus model downloads.")
print("Random-label task:", RANDOM_LABEL_TASK, "| points:", POINTS)

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY`. The
notebook looks in the environment first, then in a `.env` file at the root of
the checkout, then in Colab's secrets panel. On a rented GPU, export them
before starting Jupyter or set them in the pod's environment. Their values are
never printed or saved.

A missing one stops the notebook here. That is deliberate: the alternative is
finding out at the first upload, hours into a paid session.

In [ ]:
def secret(name):
    """Read a credential from the environment, a .env file, or Colab secrets.

    The environment comes first so that one way of supplying these works
    everywhere. On a rented GPU that is an exported variable or a pod
    template's environment; on Colab it is the secrets panel. A missing one
    stops the notebook here rather than at the first upload, hours in.
    """
    value = os.environ.get(name)
    if not value:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    value = line.partition("=")[2].strip().strip('"').strip("'")
                    break
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError(
            f"Missing secret: {name}. Export it before starting Jupyter, put "
            f"it in {REPO / '.env'}, or add it to Colab's secrets panel.")
    return value


store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))

# The settings the streams were trained with, read back rather than retyped, so
# the model this notebook rebuilds has the shape its checkpoint expects.
plan = json.loads(store.download(
    f"uncle/indicators/{STUDY_ID}/job2_summary.json", CACHE).read_text(encoding="utf-8"))
CLASSES = plan["classes_per_task"]
assert CLASSES == CLASSES_HINT, f"settings assume {CLASSES_HINT}, the runs used {CLASSES}"
print("Streams used learning rate", plan["learning_rate"], "| epochs", plan["epochs"],
      "| batch", plan["batch_size"], "| classes per task", CLASSES)

# Stop now if a stream this session is about to read has no end-of-stream
# model, rather than an hour in.
FORGET = {"order_A": "0", "order_B": "4", "order_C": "8"}
for order, seed in RUNS:
    for point in POINTS:
        if point == 0:
            continue
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        if store.info(key) is None:
            raise RuntimeError(f"Missing {key}")
print("Every model this session needs is in R2.")

## 4. Build the images and draw the labels once

In [ ]:
import math
from torch.utils.data import Subset

STREAM_TASKS = [str(i) for i in range(9)]
needed = tuple(STREAM_TASKS + [RANDOM_LABEL_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

if RANDOM_LABEL_TASK in STREAM_TASKS:
    raise ValueError("The random-label task is in the learning stream, so it is not held out.")


def balanced_split(labels, counts, seed):
    """Row numbers for disjoint splits, equal numbers from every class.

    Works on the labels alone, so no image is read here.
    """
    by_class = {}
    for row, label in enumerate(labels):
        by_class.setdefault(int(label), []).append(row)
    if len(by_class) != CLASSES:
        raise RuntimeError(f"Expected {CLASSES} classes, found {len(by_class)}.")
    stream = torch.Generator().manual_seed(seed)
    splits = [[] for _ in counts]
    for label, rows in sorted(by_class.items()):
        if len(rows) < sum(counts):
            raise RuntimeError(f"Class {label} has {len(rows)} images, needs {sum(counts)}.")
        order = [rows[j] for j in torch.randperm(len(rows), generator=stream).tolist()]
        at = 0
        for which, how_many in enumerate(counts):
            splits[which] += order[at:at + how_many]
            at += how_many
    taken = [row for split in splits for row in split]
    assert len(set(taken)) == len(taken), "An image landed in two splits."
    return splits


# The same 2,000 images at every probe point, taken evenly from the twenty
# classes. Their true labels are discarded a line below; the balance matters
# only so that the set is the same pictures every time.
per_class, remainder = divmod(RANDOM_LABEL_IMAGES, CLASSES)
if remainder:
    raise ValueError(f"{RANDOM_LABEL_IMAGES} images do not divide evenly over {CLASSES} classes.")
base = tasks[RANDOM_LABEL_TASK]["train"]
rows, = balanced_split(base.targets, (per_class,), SPLIT_SEED)
images = Subset(base, rows)

# One draw of wrong labels, shared by every probe point and every repetition.
# A fresh draw per point would make the points incomparable, because an easier
# draw would read as more capacity to change.
random_label_set = RandomLabels(images, CLASSES, RANDOM_LABEL_SEED)

CHANCE_LOSS = math.log(CLASSES)
print(f"{len(random_label_set)} images, {per_class} per class, labels from seed "
      f"{RANDOM_LABEL_SEED}")
print(f"Chance for {CLASSES} classes is a loss of ln({CLASSES}) = {CHANCE_LOSS:.3f}. "
      f"The fit has to land below {CHANCE_LOSS - CHANCE_MARGIN:.3f} to mean anything.")

## 5. Building a model at one probe point

In [ ]:
def build(order, seed, point):
    """The model this stream had at `point`. Point 0 needs no file."""
    settings = study_defaults(order)
    settings["requests"] = settings["requests"] + (("forget", FORGET[order]),)
    config = Config(**settings, seed=seed,
                    learning_rate=plan["learning_rate"], epochs=plan["epochs"],
                    batch_size=plan["batch_size"],
                    initialization=plan["initialization"], device=DEVICE)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if point > 0:
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        payload = torch.load(store.download(key, CACHE), map_location=DEVICE,
                             weights_only=False)
        checkpoint.restore(payload, hypernet=hypernet, uncle=trainer)
        del payload
    return target, hypernet, trainer

## 6. Run the grid

Every run forks the model, trains a fresh task code on the shuffled labels for
600 steps, records how far the training loss fell, and puts the model back. The
cell checks after each run that the model really is unchanged and stops if it
is not.

There is no test set and no validation set. A score on held-out images means
nothing when the labels are random, and the learning rate is chosen afterwards
on the training loss itself.

After the first stream finishes both of its points, the cell checks that the
probe actually memorised something. If the fit sat at chance the budget is too
small and every later run would measure the budget rather than the model, so
the session stops there rather than spending three hours on a void result.

In [ ]:
def identity(run):
    """Everything that makes one run different from another."""
    return (run["order"], run["seed_label"], run["point"], run["task"],
            run["learning_rate"], run["arm"], run["repetition"])


def probe_seed(task, repetition):
    """Fixes the batch order and the fresh code.

    Depends on the task and the repetition, never on the probe point, so two
    points differ only by the state of the model being probed. Repetitions have
    to differ in the starting code as well as the batch order, or the spread
    they measure is too small and the floor built from it lets anything
    through.
    """
    return 12345 + 1000 * repetition + (int(task) if task.isdigit() else 99)


PARTIAL = CACHE / "random_labels_partial.jsonl"
results = []
if store.info(RESUME_KEY) is not None:
    saved = store.download(RESUME_KEY, CACHE)
    for line in saved.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    print(f"Resuming: {len(results)} runs already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")
done = {identity(r) for r in results}
new_runs = 0


def losses_at(order, seed, point):
    """One final training loss per repetition, after the rate is chosen."""
    out = []
    for rep in range(max(REPETITIONS_AT_ZERO, REPETITIONS_LATER)):
        runs = [r for r in results
                if r["order"] == order and r["seed_label"] == seed
                and r["point"] == point and r["task"] == RANDOM_LABEL_TASK
                and r["arm"] == "full" and r["repetition"] == rep]
        if runs:
            out.append(best_of_grid(runs, endpoint="E1")["final_train_loss"])
    return out


def check_memorised(order, seed):
    """Stop the session if the probe never memorised anything.

    A final loss at chance says the step budget was too small. Nothing below
    this point can be read, so there is no reason to keep paying for it.
    """
    for point in POINTS:
        values = losses_at(order, seed, point)
        mean = sum(values) / len(values)
        print(f"  gate: {order} seed {seed}, t = {point}: mean final loss "
              f"{mean:.4f} against chance {CHANCE_LOSS:.3f}")
        if mean >= CHANCE_LOSS - CHANCE_MARGIN:
            raise RuntimeError(
                f"The probe ended near chance at t = {point} ({mean:.4f}). "
                f"It measured the step budget, not the model. Raise "
                f"RANDOM_LABEL_STEPS or lower RANDOM_LABEL_IMAGES, give the "
                f"run a new RUN_LABEL, and start again.")
    print("  gate passed: the probe memorised on both points of the first stream.")


def grid_at(order, seed, point, repetitions):
    """Every rate, arm and repetition at one point of one stream."""
    global new_runs
    todo = [(rate, arm, code_only, train_code, rep)
            for rate in RATES
            for arm, code_only, train_code in ARMS
            for rep in range(repetitions)
            if (order, seed, point, RANDOM_LABEL_TASK, rate, arm, rep) not in done]
    if not todo:
        print(f"{order} seed {seed}, t = {point}: already finished")
        return

    target, hypernet, trainer = build(order, seed, point)
    reference = snapshot(hypernet, trainer)
    bar = tqdm(todo, desc=f"{order} seed {seed}  t={point}", unit="run")
    for rate, arm, code_only, train_code, rep in bar:
        bar.set_postfix_str(f"lr={rate:g} rep {rep + 1}")
        started = time.perf_counter()
        run = probe_once(
            hypernet, target, trainer, task=RANDOM_LABEL_TASK,
            train_set=random_label_set, test_set=None, validation_set=None,
            steps=RANDOM_LABEL_STEPS, learning_rate=rate, batch_size=BATCH_SIZE,
            code_only=code_only, train_code=train_code,
            seed=probe_seed(RANDOM_LABEL_TASK, rep), feature_images=0)
        # A probe forks and discards. If it changed the model, every later
        # reading in this session is measuring something that has moved.
        if not unchanged(hypernet, reference):
            raise RuntimeError("A probe changed the model. Stop and fix this "
                               "before trusting anything.")
        run.update(order=order, point=point, endpoint="E1", repetition=rep,
                   seed_label=seed, minutes=(time.perf_counter() - started) / 60,
                   device=GPU_NAME, run_label=RUN_LABEL, code_revision=HEAD,
                   label_seed=RANDOM_LABEL_SEED, train_images=len(random_label_set))
        results.append(run)
        done.add(identity(run))
        with PARTIAL.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(run) + "\n")
        new_runs += 1
        if new_runs % SAVE_EVERY == 0:
            store.upload(PARTIAL, RESUME_KEY, overwrite=True)
    bar.close()

    del trainer, hypernet, target
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    store.upload(PARTIAL, RESUME_KEY, overwrite=True)


# The gate has already been answered if a previous session got past the first
# stream, so a resumed run does not pay for it again.
first = RUNS[0]
gate_checked = all(losses_at(*first, point) for point in POINTS)
if gate_checked:
    print("Gate already passed in an earlier session.")

began = time.perf_counter()
for order, seed in RUNS:
    for point in POINTS:
        grid_at(order, seed, point,
                REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER)
    if not gate_checked and (order, seed) == first:
        check_memorised(order, seed)
        gate_checked = True
print(f"\nFinished in {(time.perf_counter() - began) / 3600:.1f} hours. "
      f"{len(results)} runs in total.")

## 7. The result

In [ ]:
import statistics

lo, hi = POINTS[0], POINTS[-1]
WIDTH = 86


def rule(title):
    print(f"\n{title}\n" + "-" * WIDTH)


print("=" * WIDTH)
print(f"RANDOM-LABEL FITTING, {STUDY_ID}, {', '.join(ORDERS)}".center(WIDTH))
print(f"{len(results)} runs | task {RANDOM_LABEL_TASK}, {CLASSES} classes, "
      f"labels from seed {RANDOM_LABEL_SEED}".center(WIDTH))
print("=" * WIDTH)

rule("TABLE 1. Final training loss after choosing the best rate, one value per repetition")
print(f"{'stream':>16}{'point':>7}{'reps':>6}{'mean':>10}{'sd':>9}  values")
print("-" * WIDTH)
for order, seed in RUNS:
    for point in POINTS:
        values = losses_at(order, seed, point)
        if not values:
            continue
        sd = statistics.stdev(values) if len(values) > 1 else float("nan")
        print(f"{order + '_seed' + str(seed):>16}{point:>7}{len(values):>6}"
              f"{statistics.mean(values):>10.4f}{sd:>9.4f}  "
              + " ".join(f"{v:.4f}" for v in values))
print("-" * WIDTH)
print("This is training loss on 2,000 images whose labels were shuffled. LOWER means the")
print("model memorised more of them, so LOWER means MORE capacity to change, not less.")

rule("TABLE 2. The decision rule, one row per stream")
print(f"{'stream':>16}{'t=' + str(lo):>9}{'t=' + str(hi):>9}{'change':>10}"
      f"{'floor':>9}{'exceeds':>9}  verdict")
print("-" * WIDTH)
rows = []
for order, seed in RUNS:
    zero, later = losses_at(order, seed, lo), losses_at(order, seed, hi)
    if len(zero) < 2 or len(later) < 2:
        print(f"{order + '_seed' + str(seed):>16}  not finished")
        continue
    mean_zero, mean_later = statistics.mean(zero), statistics.mean(later)
    change = mean_later - mean_zero
    spread = statistics.stdev(zero)
    floor = noise_floor(spread, spread, len(zero), len(later))
    exceeds = abs(change) > floor
    # Loss is a RISE here, because the number is a training loss: a model that
    # can no longer memorise ends higher. A fall is the opposite of plasticity
    # loss.
    valid = all(statistics.mean(v) < CHANCE_LOSS - CHANCE_MARGIN for v in (zero, later))
    verdict = ("VOID, the probe never memorised" if not valid else
               "PLASTICITY LOSS" if exceeds and change > 0 else
               "no loss, more capacity to change" if exceeds else
               "NO FINDING, inside the floor")
    rows.append({"order": order, "seed": seed, "t0": mean_zero, "t_end": mean_later,
                 "change": change, "noise_floor": floor, "exceeds_floor": exceeds,
                 "probe_valid": valid, "verdict": verdict})
    print(f"{order + '_seed' + str(seed):>16}{mean_zero:>9.4f}{mean_later:>9.4f}"
          f"{change:>+10.4f}{floor:>9.4f}{('yes' if exceeds else 'no'):>9}  {verdict}")
print("-" * WIDTH)
print(f"Chance for {CLASSES} classes is ln({CLASSES}) = {CHANCE_LOSS:.3f}. Every mean above")
print("should sit well below it, or the run measured the step budget and not the model.")

rule("TABLE 3. Across streams. This is the headline, not any single row above")
if rows:
    changes = [r["change"] for r in rows]
    sd = statistics.stdev(changes) if len(changes) > 1 else float("nan")
    fell = sum(r["change"] < -r["noise_floor"] for r in rows)
    rose = sum(r["change"] > r["noise_floor"] for r in rows)
    print(f"{len(rows)} streams, mean change {statistics.mean(changes):+.4f}, sd {sd:.4f}")
    print("per stream: " + " ".join(f"{c:+.4f}" for c in changes))
    print(f"Fell past its floor in {fell} of {len(rows)}, rose past it in {rose}.")
    print()
    if fell == len(rows):
        print("Every stream memorised more at the end of the stream than at the start.")
        print("Capacity to change rose while the rank indicators fell, which is the")
        print("same disagreement the direct probe reports, now on a measurement that")
        print("has nothing to transfer.")
    elif rose == len(rows):
        print("Every stream memorised less at the end. That is what a loss of")
        print("plasticity looks like, and it would disagree with the direct probe on")
        print("these same models. Read both before concluding anything.")
    else:
        print("The streams do not agree with each other. Report the split as it is.")
print("-" * WIDTH)

summary = {"study": STUDY_ID, "run_label": RUN_LABEL, "code_revision": HEAD,
           "task": RANDOM_LABEL_TASK, "classes": CLASSES, "points": [lo, hi],
           "label_seed": RANDOM_LABEL_SEED, "images": RANDOM_LABEL_IMAGES,
           "steps": RANDOM_LABEL_STEPS, "rates": RATES,
           "repetitions": REPETITIONS_AT_ZERO, "chance_loss": CHANCE_LOSS,
           "rows": rows}
path = CACHE / "random_labels_summary.json"
path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
store.upload(path, f"uncle/random_labels/{STUDY_ID}/{RUN_LABEL}_summary.json",
             overwrite=True)
print("Written:", path, "and uploaded to R2.")

## 8. What this leaves

If every stream falls past its floor, random-label fitting is measured on the
same models as the five indicators, the direct probe and the curvature arms.
The row in the paper stops carrying its "different training run" caveat, and
the comparison between the no-training diagnostics and the direct measurements
sits on one set of models throughout.

Three things it does not settle.

| Question | Where it stands |
| --- | --- |
| Is the rate choice fair? | The rate is picked on the same loss that is reported. The same rule runs at both points, so the comparison between them holds, but the number is a best-achievable fit rather than a score. |
| Does it hold across orderings? | This run asks it. Ordering A already fell past its floor on all five streams; if B and C do the same, the result does not depend on the order the groups arrive in. |
| Is 600 steps the right budget? | The gate in section 6 only checks that the fit landed below chance. A longer budget would let every model memorise more, and whether the gap between the points survives that is untested. |